# 🐟 TRELLIS Batch Fish Model Generator

Converts Aquadex species PNGs into 3D GLB models using Microsoft TRELLIS.

**Instructions:**
1. Go to Runtime → Change runtime type → Select **T4 GPU** (free) or **A100** (Pro)
2. Upload your species PNGs to the `input_images/` folder (or connect Google Drive)
3. Run all cells
4. Download the generated GLB files from `output_models/`
5. Place them in `frontend/public/models/fish/` in your project

Each model takes ~30-90 seconds on a T4 GPU.

In [ ]:
# Cell 1: Install TRELLIS
!git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
%cd TRELLIS
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q pillow imageio trimesh xformers spconv-cu120 kaolin
!pip install -q huggingface_hub gradio
# Install submodules
!pip install -q git+https://github.com/JeffreyXiang/diffoctreerast.git
!pip install -q git+https://github.com/MaxtirError/FlexiCubes.git
!pip install -q nvdiffrast
print('✅ Installation complete')

In [ ]:
# Cell 2: Create folders and upload images
import os
os.makedirs('input_images', exist_ok=True)
os.makedirs('output_models', exist_ok=True)

# Option A: Upload files manually
from google.colab import files
print('Upload your species PNG files (e.g., betta-splendens.png):')
print('Or skip this cell and use Google Drive instead (see next cell)')
uploaded = files.upload()
for filename in uploaded:
    os.rename(filename, f'input_images/{filename}')
    print(f'  ✓ {filename}')

In [ ]:
# Cell 2b (Alternative): Mount Google Drive
# If you put your PNGs in a Drive folder, mount and copy them

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/fish-images/*.png input_images/
# print(f'Copied {len(os.listdir("input_images"))} images')

In [ ]:
# Cell 3: Load TRELLIS pipeline
import os
os.environ['SPCONV_ALGO'] = 'native'
os.environ['ATTN_BACKEND'] = 'xformers'  # Use xformers for T4 compatibility

from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
from PIL import Image

# Load the model (downloads ~4GB on first run)
print('Loading TRELLIS-image-large model (this takes 2-3 minutes first time)...')
pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()
print('✅ Model loaded and ready')

In [ ]:
# Cell 4: Batch generate 3D models from all PNGs
import time
from pathlib import Path

input_dir = Path('input_images')
output_dir = Path('output_models')

image_files = sorted(input_dir.glob('*.png'))
print(f'Found {len(image_files)} species images to process\n')

results = []

for i, img_path in enumerate(image_files):
    slug = img_path.stem  # e.g., 'betta-splendens'
    output_path = output_dir / f'{slug}.glb'
    
    # Skip if already generated
    if output_path.exists():
        print(f'  [{i+1}/{len(image_files)}] ⏭️  {slug} (already exists)')
        results.append({'slug': slug, 'status': 'skipped'})
        continue
    
    print(f'  [{i+1}/{len(image_files)}] 🐟 Generating {slug}...', end=' ')
    start = time.time()
    
    try:
        # Load and prepare image
        image = Image.open(img_path).convert('RGBA')
        
        # Generate 3D
        outputs = pipeline.run(
            image,
            seed=42,
            sparse_structure_sampler_params={'steps': 12, 'cfg_strength': 7.5},
            slat_sampler_params={'steps': 12, 'cfg_strength': 3},
        )
        
        # Export as GLB with simplification for game-ready mesh
        glb = postprocessing_utils.to_glb(
            outputs['gaussian'][0],
            outputs['mesh'][0],
            simplify=0.95,        # Remove 95% of triangles (aggressive but fine for fish)
            texture_size=1024,    # 1K texture (good balance of quality/size)
        )
        glb.export(str(output_path))
        
        elapsed = time.time() - start
        size_kb = output_path.stat().st_size / 1024
        print(f'✅ ({elapsed:.1f}s, {size_kb:.0f} KB)')
        results.append({'slug': slug, 'status': 'success', 'time': elapsed, 'size_kb': size_kb})
        
    except Exception as e:
        print(f'❌ Error: {e}')
        results.append({'slug': slug, 'status': 'error', 'error': str(e)})

# Summary
print(f'\n{"="*50}')
success = [r for r in results if r['status'] == 'success']
errors = [r for r in results if r['status'] == 'error']
print(f'✅ Generated: {len(success)}')
print(f'⏭️  Skipped: {len([r for r in results if r["status"] == "skipped"])}')
print(f'❌ Errors: {len(errors)}')
if success:
    avg_time = sum(r['time'] for r in success) / len(success)
    print(f'⏱️  Average time: {avg_time:.1f}s per model')

In [ ]:
# Cell 5: Download all generated models
import shutil

# Zip all GLBs for easy download
shutil.make_archive('fish_models', 'zip', 'output_models')
print(f'📦 Created fish_models.zip with {len(list(output_dir.glob("*.glb")))} models')

# Download the zip
from google.colab import files
files.download('fish_models.zip')
print('\n📥 Download started!')
print('\nNext steps:')
print('1. Unzip fish_models.zip')
print('2. Copy all .glb files to: frontend/public/models/fish/')
print('3. Refresh the Immersive Reef — 3D models load automatically!')

In [ ]:
# Cell 6 (Optional): Preview a generated model
import trimesh
from IPython.display import HTML

# Pick the first generated model to preview
glb_files = sorted(output_dir.glob('*.glb'))
if glb_files:
    mesh = trimesh.load(str(glb_files[0]))
    print(f'Preview: {glb_files[0].name}')
    print(f'  Vertices: {sum(m.vertices.shape[0] for m in mesh.geometry.values() if hasattr(m, "vertices"))}')
    print(f'  Faces: {sum(m.faces.shape[0] for m in mesh.geometry.values() if hasattr(m, "faces"))}')
    # Show inline 3D viewer
    scene = mesh.scene() if hasattr(mesh, 'scene') else mesh
    HTML(scene.show())
else:
    print('No models generated yet. Run Cell 4 first.')